<a href="https://colab.research.google.com/github/brpetros/prompts_and_evalution_notebooks/blob/main/1_skillab_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Log in and functions definition

In [ ]:
import requests
from pprint import pprint
from google.colab import userdata
import os

BASE_URL = "https://skillab-tracker.csd.auth.gr/api"

def login(username, password):
    url = f"{BASE_URL}/login"

    response = requests.post(url, json={"username": username, "password": password}, verify=False)
    response.raise_for_status()
    return response.json()

os.environ['API_TOKEN']= login(userdata.get("SKILLAB_USERNAME"), userdata.get("SKILLAB_PASSWORD"))

def auth_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type" : "application/x-www-form-urlencoded",
        "Accept": "application/json"
    }


def get_occupations(token, keywords=None, ancestors=None, keywords_logic="or"):
    url = f"{BASE_URL}/occupations"
    all_items = []
    page = 1
    page_size = 100
    payload = {}
    if keywords is not None:
        payload["keywords"] = keywords
        payload["keywords_logic"] = keywords_logic
    if ancestors is not None:
        payload["ancestors"] = ancestors

    print(payload)

    while True:
        params = {"page": page, "page_size": page_size}
        response_data = requests.post(url, params=params, data=payload, headers=auth_headers(token), verify=False).json()

        if not response_data.get('items'):
            break

        all_items.extend(response_data['items'])
        page += 1
    return {"count": len(all_items), "items": all_items}

# break an array into pieces
def chunked(iterable, size):
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]


def get_jobs(token, keywords, occupations_data, keywords_logic="or"):
    url = f"{BASE_URL}/jobs"
    jobs = {}

    page_size = 100
    CHUNK_SIZE = 25
    MAX_PAGES = 1000
    occupation_ids = list({occupation.get('id') for occupation in occupations_data.get('items')})

    for occ_chunk in chunked(occupation_ids, CHUNK_SIZE):
        page = 1

        while page <= MAX_PAGES:
            params = {"page": page, "page_size": page_size}
            payload = {"keywords": keywords, "keywords_logic": keywords_logic, "occupation_ids": occ_chunk}

            response = requests.post(
                url,
                params=params,
                data=payload,
                headers=auth_headers(token),
                verify=False
            )
            response.raise_for_status()
            data = response.json()

            items = data.get("items", [])
            if not items:
                break

            for job in items:
                jobs[job["id"]] = job

            page += 1

    return {
        "count": len(jobs),
        "items": list(jobs.values())
    }





/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


# Searching for the agriculture related occupations using keywords



In [ ]:

keywords = ["agriculture", "agro-food",
                            "agronomy", "farming",
                            "horticulture", "livestock", "dairy",
                            "fisheries", "food processing", "crop management",
                            "sustainable agriculture", "organic farming" ]

if os.environ.get('API_TOKEN'):
    occupations_data = get_occupations(token=os.environ.get('API_TOKEN'), keywords=keywords)
    pprint(occupations_data)

else:
    print("Failed to obtain access token.")

{'keywords': ['agriculture', 'agro-food', 'agronomy', 'farming', 'horticulture', 'livestock', 'dairy', 'fisheries', 'food processing', 'crop management', 'sustainable agriculture', 'organic farming'], 'keywords_logic': 'or'}


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'count': 105,
 'items': [{'alternative_labels': ['fresh fish salesperson',
                                   'fish market salesperson',
                                   'fish and seafood specialized seller',
                                   'fresh seafood specialised seller',
                                   'fresh fish counter specialised seller',
                                   'fish stall specialised seller',
                                   'fish salesperson',
                                   'fresh fish specialised seller',
                                   'fresh seafood salesperson',
                                   'fish market specialised seller',
                                   'seafood specialised seller',
                                   'fisheries shop salesperson',
                                   'fresh fish shop salesperson',
                                   'seafood salesperson',
                                   'fish specialised seller',
 

we notice that the occupations that are returned belong to different ISCO categories, not only to ISCO category 6, which is for Skilled Agricultural, Forestry and Fishery Workers. Nevertheless, many of them are relevant. For example, we get an occupation with the label 'livestock worker', which is relevant. This belongs to ISCO category 9 (Elementary Occupations).

So our next step is to search for any category related specifically to Agriculture in the ISCO classification. We use the tables at: https://ilostat.ilo.org/methods/concepts-and-definitions/classification-occupation/ for that matter.

The categories that we found are the following:

| ISCO code | Description |
| --------- | ----------- |
| 131 | Production Managers in Agriculture, Forestry and Fisheries |
| 2131 | Biologists, Botanists, Zoologists and Related Professionals |
| 2132 | Farming, Forestry and Fisheries Advisers |
| 3142 | Agricultural Technicians |
| 6 | Skilled Agricultural, Forestry and Fishery Workers |
| 7233 | Agricultural and Industrial Machinery Mechanics and Repairers |
| 816 | Food and Related Products Machine Operators |
| 92 |  Agricultural, Forestry and Fishery Labourers |


Intentionally, we are very specific on some categories, and more broad on some others. This is because category 6 is dedicated to Agricultural Workers, so we can take all the occupations and exclude manually what is irrelevant. On the contrary, categories like 2 (Professionals) are too generic to take all the occupations, so we focus only on the more specific sub-categories.

In any case, we can manually add or exclude occupations based on the relevance of the results.


# Extraction of agriculture related occupations

We are now going to find all the occupations that have the above ISCO categories as ancestors.  




In [ ]:
ancestors = [ 'http://data.europa.eu/esco/isco/C131',


              'http://data.europa.eu/esco/isco/C3142',
              'http://data.europa.eu/esco/isco/C6',
              'http://data.europa.eu/esco/isco/C7233',
              'http://data.europa.eu/esco/isco/C816',
              'http://data.europa.eu/esco/isco/C92']

if os.environ.get('API_TOKEN'):
    occupations_data = get_occupations(token=os.environ.get('API_TOKEN'), ancestors=ancestors)
    for occupation in occupations_data['items']:
        print(occupation.get('label'))
    print(len(occupations_data['items']))
    #pprint(occupations_data)

else:
    print("Failed to obtain access token.")


{'ancestors': ['http://data.europa.eu/esco/isco/C131', 'http://data.europa.eu/esco/isco/C2131', 'http://data.europa.eu/esco/isco/C2132', 'http://data.europa.eu/esco/isco/C3142', 'http://data.europa.eu/esco/isco/C6', 'http://data.europa.eu/esco/isco/C7233', 'http://data.europa.eu/esco/isco/C816', 'http://data.europa.eu/esco/isco/C92']}


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-

hydrogenation machine operator
pasta operator
coffee grinder
land-based machinery technician
bee breeder
aquaculture hatchery manager
fluid power technician
water-based aquaculture technician
fur animals breeder
candy machine operator
pharmacologist
chilling operator
blending plant operator
horse breeder
sauce production operator
tree surgeon
biophysicist
horticulture worker
crop production manager
mixed farmer
brew house operator
agricultural technician
vineyard machinery operator
on foot aquatic resources collector
vineyard supervisor
greaser
rotating equipment mechanic
deep-sea fishery worker
physiologist
industrial machinery mechanic
fisheries boatman
centrifuge operator
shepherd
textile machinery technician
aquaculture cage mooring worker
farm manager
aquaculture cage technician
sugar refinery operator
cocoa press operator
coffee roaster
fruit and vegetable picker
starch converting operator
fisheries adviser
kettle tender
forestry adviser
cellar operator
toxicologist
aquaculture h

# Limiting the results

We notice that using the above keywords, although the results are importantly limited, some of the job offers that are found are not significantly relevant to agriculture.

It is important that we keep only the agriculture related jobs, so that out data is accurate and worthy of analysis.


## Method

To face the above problem, we will reduce the ESCO categories used as parameters. In addition, we will perform multiple searches with different keywords each time. All the retrieved items (job items) will be stored in a dictionnary with the job id as the id.

Finally, we will perform manual removal of any job offer that is not related to agriculture.

We understand that this could potentially exclude some of the job offers, even some that are actually relevant to agriculture. Nevertheless, as thousands of job offers exist, we consider that the final statistical outcome is accurate.

We will modify our methods if needed.

## Modifications

From the ISCO table used previously, decided to exclude:

- 2131: Biologists, Botanists, Zoologists and Related Professionals

  It is too broad and specialized in a medical way, it contains skills that are not relevant or not necessary for an agriculture-related profession

- 7233: Agricultural and Industrial Machinery Mechanics and Repairers

  It concernes mechanics and agricultural staff/managers. Also, it concerns industrial machinery in general so it is also too broad.

-  816: Food and Related Products Machine Operators

Therefore, the categories we are eventually using are the following:


| ISCO code | Description |
| --------- | ----------- |
| 131 | Production Managers in Agriculture, Forestry and Fisheries |
| 2132 | Farming, Forestry and Fisheries Advisers |
| 3142 | Agricultural Technicians |
| 6 | Skilled Agricultural, Forestry and Fishery Workers |
| 92 |  Agricultural, Forestry and Fishery Labourers |

In addition, we will perform multiple searches

Getting the skills found in the job offers

In [ ]:
ancestors = [ 'http://data.europa.eu/esco/isco/C131',
            'http://data.europa.eu/esco/isco/C2132',
            'http://data.europa.eu/esco/isco/C3142',
            'http://data.europa.eu/esco/isco/C6',
            'http://data.europa.eu/esco/isco/C92']

#ancestors = ['http://data.europa.eu/esco/isco/C92']

if os.environ.get('API_TOKEN'):
    occupations_data = get_occupations(os.environ.get('API_TOKEN'), ancestors=ancestors)
    for occupation in occupations_data['items']:
        print(occupation.get('label'))
    print(len(occupations_data['items']))
    #pprint(occupations_data)

else:
    print("Failed to obtain access token.")


{'ancestors': ['http://data.europa.eu/esco/isco/C131', 'http://data.europa.eu/esco/isco/C2132', 'http://data.europa.eu/esco/isco/C3142', 'http://data.europa.eu/esco/isco/C6', 'http://data.europa.eu/esco/isco/C92']}


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


bee breeder
aquaculture hatchery manager
water-based aquaculture technician
fur animals breeder
horse breeder
tree surgeon
horticulture worker
crop production manager
mixed farmer
agricultural technician
vineyard machinery operator
on foot aquatic resources collector
vineyard supervisor
deep-sea fishery worker
fisheries boatman
shepherd
aquaculture cage mooring worker
farm manager
aquaculture cage technician
fruit and vegetable picker
fisheries adviser
forestry adviser
aquaculture harvesting manager
equine yard manager
aquaculture production manager
fruit production team leader
livestock worker
equine worker
forest worker
aquaculture rearing technician
poultry sexer
aquaculture husbandry manager
vineyard manager
horticulture production team leader
interior landscaper
garden labourer
catcher
aquaculture recirculation manager
groundsman/groundswoman
hop farmer
aquaculture hatchery worker
water-based aquaculture worker
aquaculture site supervisor
sheep breeder
cattle breeder
arboriculturi

We now get 111 occupations, before it was 203

# Keywords based on litterature's thematic map by Bibliometrix

In [ ]:
justified_keywords = [
    "agricultural management",
    "farm management",
    "crop management"
    "sustainable agriculture",
    "food security",
    "biodiversity",
    "cropping systems",
    "land-use",
    "crop production",
    "conservation agriculture",
    "farming systems",
    "irrigation strategies",
    "manage livestock",
    "soil fertility",
    "agroecology",
    "soil organic matter",
    "water management",
    "soil health",
    "tillage",
    "no-tillage",
    "crop resilience",
    "crop yield",
    "deficit irrigation",
    "nitrogren use efficiency",
    "water productivity",
    "fertilizer use",
    "drip irrigation",
    "precision agriculture",
    "precision farming",
    "agriculture remote sensing",
    "smart agriculture",
    "crop disease",
    "pest management",
    "weed detection",
    "grain yield",
    "efficient water use",
    "animal breeding",
    "biofortification",
    "crop improvement",
    "plant breeding",
    "powdery mildew",
    "integrated pest management",
    "biological control",
    "weed management",
    "use pesticides",
    "integrated weed management",
    "crop protection"
]

print(len(justified_keywords))

46


In [ ]:

jobs_data = get_jobs(os.environ.get('API_TOKEN'), justified_keywords, occupations_data, "or")
# pprint(jobs_data)
for job in jobs_data['items']:
    print(job.get('title'))
print(f"successful access to jobs data - count: {jobs_data.get("count")}")





/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'skillab-tracker.csd.auth.gr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-

MANAGER OF PRODUCTION
Agricultural Sales - Export Managers
Farm Veterinarian
Veterinary
Senior Consultant Agriculture and Food Sector F/M
DIGITAL TRANSFORMATION INNOVATION PROJECT MANAGER M/F
Technical Leader in boilermaking H/F
Sales consultant (m/f/d) agricultural and construction machinery
Production Team Leader 3*8 M/F
Responsable QSE H/F
Agro Logistics Manager M/F
Agro Logistics Manager
Agricultural workshop manager
Agricultural mechanic M/F
Versatile agricultural worker M/F
Poultry worker M/F
Poultry worker M/F
Landscape gardener M/F
Agricultural Technician M/F
Land and Agricultural Estate Operations Director M/F
Security Facilitator M/F
Water Operations Technician M/F
Day Maintenance Technician M/F
Maintenance and New Works Technician M/F
After-sales team leader M/F
Technician, Animal Research Facility [LKCMedicine]
Technician, Animal Research Facility [LKCMedicine]
LABORATORY ANIMAL
LABORATORY ANIMAL
Sales Representative
WAREHOUSE & CUSTOMER MANAGEMENT OFFICER (FARMER / FOOD TE

1681 returned jobs

## Filter jobs

Many of the job offers, although they contain the keywords and the occupations we searched for, they are not relevant to agriculture.

We are going to filter them based on the occupations labels that we stored above.





## filtering according to skills

We already have a set of skills related (optionally or significantly) to agriculture extracted by ESCO database.
We are going to exclude any job offer that has no skills that belong to that set

In [ ]:
import pandas as pd

try:
  df = pd.read_csv("/content/EssentialAgriculturalSkills.csv")

  relevant_esco_skills = df.set_index('skill').to_dict(orient='index')
except FileNotFoundError:
  print("File not found.")

def filter_according_to_skills(jobs_data, relevant_skills):
    accepted = {}
    rejected = {}
    for job in jobs_data.get('items'):
      for skill_id in job.get('skills'):
        if skill_id in relevant_skills:
          accepted[job['id']] = job
          break
      if job['id'] not in accepted.keys():
        rejected[job['id']] = job

    return accepted, rejected

accepted, rejected = filter_according_to_skills(jobs_data, relevant_esco_skills)
print_filter_results(accepted, rejected)

Total number of accepted jobs: 971
Total number of rejected jobs: 710
--- Accepted Jobs ---
MANAGER OF PRODUCTION - 1659
Agricultural Sales - Export Managers - 2638
Farm Veterinarian - 2893
Veterinary - 3237
Senior Consultant Agriculture and Food Sector F/M - 6744
Agro Logistics Manager M/F - 20230
Agro Logistics Manager - 22502
Agricultural workshop manager - 26782
Agricultural mechanic M/F - 26783
Versatile agricultural worker M/F - 28966
Poultry worker M/F - 29242
Poultry worker M/F - 29282
Landscape gardener M/F - 30088
Agricultural Technician M/F - 30540
Land and Agricultural Estate Operations Director M/F - 31908
Day Maintenance Technician M/F - 32661
Maintenance and New Works Technician M/F - 32677
Technician, Animal Research Facility [LKCMedicine] - 37758
Technician, Animal Research Facility [LKCMedicine] - 37759
LABORATORY ANIMAL - 40398
LABORATORY ANIMAL - 40542
Agronomic R&D Technician | Agronomist - 45646
Agronomic R&D Technician | Agronomist - 45647
Agronomic R&D Technicia

In [ ]:
a,b,c = analyze_job_result(rejected ,justified_keywords, occupations_data, job_title="Cattle keeper, in agriculture")

--- Selected Job ---
{'description': 'Prästbol is a modern agricultural company in Södra Värmland. '
                'Here, beef production and plant cultivation are carried out '
                'on roughly 500 ha of land. In addition, there are some '
                'natural pastures and 100 ha of forest. We are looking for an '
                'employee for agriculture, where various work is included. You '
                'need to have at least 3 years of experience working with '
                'cattle and have an interest in animal handling. You also need '
                'to have an interest in crop cultivation and be able to drive '
                'a tractor and loader. We are looking for someone who is '
                'organized and easy to work with. It is good if you know '
                'Swedish and English. The work involves varied tasks. It is '
                'important that you are the right person. A first contact '
                'should be made via email to

In [ ]:
def filter_according_to_occupations(jobs_data, occupations_data):
  accepted = {}
  rejected = {}
  occupation_ids = list({occupation.get('id') for occupation in occupations_data.get('items')})
  for job in jobs_data.values():
    relevant_occupations = []
    for occupation_id in occupation_ids:
      if occupation_id in job.get('occupations'):
        relevant_occupations.append(occupation_id)
    if len(relevant_occupations)/len(job.get('occupations')) >= 0.3:
      accepted[job['id']] = job
    else:
      rejected[job['id']] = job

  return accepted, rejected


Total number of accepted jobs: 268
Total number of rejected jobs: 703
--- Accepted Jobs ---
MANAGER OF PRODUCTION - 1659
Farm Veterinarian - 2893
Veterinary - 3237
Senior Consultant Agriculture and Food Sector F/M - 6744
Versatile agricultural worker M/F - 28966
Poultry worker M/F - 29242
Poultry worker M/F - 29282
Agricultural Technician M/F - 30540
Land and Agricultural Estate Operations Director M/F - 31908
Day Maintenance Technician M/F - 32661
Agronomic R&D Technician | Agronomist - 45646
Agronomic R&D Technician | Agronomist - 45647
Agronomic R&D Technician | Agronomist - 45653
digital domain manager F/M - 52208
Agronomic R&D Technician | Agronomist - 61744
Day Maintenance Technician M/F - 63540
Versatile agricultural worker M/F - 71654
Industrial maintenance technician M/F Domaine Avicole - 72038
Chauffeur SPL* H/F - 73928
Poultry worker M/F - 74028
Agricultural mechanic M/F - 77751
Land and Agricultural Estate Operations Director M/F - 79161
Day Maintenance Technician M/F - 803

In [ ]:
def filter_jobs(jobs, relevant_skills, occupations_data):
    accepted_by_skills , rejected_by_skills = filter_according_to_skills(jobs, relevant_skills)
    accepted_by_occupations, rejected_by_occupations = filter_according_to_occupations(accepted_by_skills, occupations_data)

    accepted = accepted_by_occupations
    rejected = rejected_by_skills | rejected_by_occupations

    return accepted, rejected


In [ ]:
accepted, rejected = filter_jobs(jobs_data, relevant_esco_skills, occupations_data)
print_filter_results(accepted, rejected)

Total number of accepted jobs: 268
Total number of rejected jobs: 1413
--- Accepted Jobs ---
MANAGER OF PRODUCTION - 1659
Farm Veterinarian - 2893
Veterinary - 3237
Senior Consultant Agriculture and Food Sector F/M - 6744
Versatile agricultural worker M/F - 28966
Poultry worker M/F - 29242
Poultry worker M/F - 29282
Agricultural Technician M/F - 30540
Land and Agricultural Estate Operations Director M/F - 31908
Day Maintenance Technician M/F - 32661
Agronomic R&D Technician | Agronomist - 45646
Agronomic R&D Technician | Agronomist - 45647
Agronomic R&D Technician | Agronomist - 45653
digital domain manager F/M - 52208
Agronomic R&D Technician | Agronomist - 61744
Day Maintenance Technician M/F - 63540
Versatile agricultural worker M/F - 71654
Industrial maintenance technician M/F Domaine Avicole - 72038
Chauffeur SPL* H/F - 73928
Poultry worker M/F - 74028
Agricultural mechanic M/F - 77751
Land and Agricultural Estate Operations Director M/F - 79161
Day Maintenance Technician M/F - 80

In [ ]:
a,b,c = analyze_job_result(accepted ,justified_keywords, occupations_data, job_title="Växthusarbetare till food tech-bolag")

--- Selected Job ---
{'description': 'Do you have a genuine interest in plants? Are you attracted '
                'by independent physical work in a lovely environment? Apply '
                "for a position as a greenhouse worker for one of Sweden's "
                'most exciting food tech companies. Your future challenge '
                'Agtira is an innovative food tech company that supplies and '
                'operates vegetable farms. Right now, the company is in a '
                'growth phase and will therefore start a new facility in '
                'Kramfors in early 2025. For this facility, we are now looking '
                'for greenhouse workers who want to be involved from the start '
                'to build up this exciting business. Within Agtira, we are a '
                'total of 31 people who are distributed across five different '
                'facilities in Sweden. We are a tightly knit team that works '
                'together to develop t

## extracting the skills

In [ ]:
def extract_skills (jobs):
  skills = {}
  for job in jobs.values():
    for skill_id in job.get('skills'):
      if skill_id not in skills:
        skills[skill_id] = [job['id']]
      else:
        skills[skill_id].append(job['id'])
  return skills


In [ ]:
skills = extract_skills(accepted)

pprint(skills)
print(f"total skills found: {len(skills)}")

{'http://data.europa.eu/esco/skill/00064735-8fad-454b-90c7-ed858cc993f2': [1788922],
 'http://data.europa.eu/esco/skill/001115fb-569f-4ee6-8381-c6807ef2527f': [123970,
                                                                           1631314,
                                                                           1725773,
                                                                           1736520,
                                                                           1752303,
                                                                           1831136,
                                                                           1831622,
                                                                           159328,
                                                                           1665735,
                                                                           123574,
                                                                           163

In [ ]:
def filter_skills(skills, relevant_esco_skills):
  filtered_skills = {}
  for skill_id, job_ids in skills.items():
    if skill_id in relevant_esco_skills:
      filtered_skills[skill_id] = {'label': relevant_esco_skills.get(skill_id).get('skillLabel'), 'job_count': len(job_ids), 'job_ids': job_ids }
  return filtered_skills

skill_dict = filter_skills(skills, relevant_esco_skills)
pprint(skill_dict)
print(f"total of {len(skill_dict)} relevant skills")

{'http://data.europa.eu/esco/skill/00dd3271-077f-4a12-a958-e297fdd724ce': {'job_count': 6,
                                                                           'job_ids': [77751,
                                                                                       141059,
                                                                                       1647466,
                                                                                       117800,
                                                                                       1625144,
                                                                                       1851304],
                                                                           'label': 'operate '
                                                                                    'agricultural '
                                                                                    'machinery'},
 'http://data.europa.eu/esco/skill/0612b3e9

# Filtering the skills

1716 skills are returned and most of them are not relevant to agriculture. So we need to filter them

We import the csv file with all the optional and essential skills related to agriculture and taking the skills column which contains the skills (the file was created using the turtle version of ESCO data and a simple sparql query to find all the skills that are optionally or essentially connected to the occupations related to agriculture - it contains 855 skills)

In [ ]:
sorted_skills = sorted(skill_dict.items(), key=lambda item: item[1]['job_count'], reverse=True)

print("--- Relevant Skills sorted by job_count (Descending) ---")
for skill_id, skill_info in sorted_skills:
    print(f"Skill ID: {skill_id}, Label: {skill_info['label']}, Job Count: {skill_info['job_count']}, Job IDs: {skill_info['job_ids']}")

--- Relevant Skills sorted by job_count (Descending) ---
Skill ID: http://data.europa.eu/esco/skill/2e4e8114-e579-453d-918e-b0262818ab76, Label: harvest crop, Job Count: 35, Job IDs: [32661, 52208, 63540, 80351, 91684, 109562, 114191, 1565152, 1698582, 1787132, 1852203, 1867000, 1880592, 22777, 31431, 57818, 59176, 61220, 63570, 67959, 78711, 83448, 1567924, 1739844, 1772492, 1774418, 1788723, 1794245, 1817157, 1819918, 1824383, 1824791, 1835897, 1943093, 1751210]
Skill ID: http://data.europa.eu/esco/skill/4ca83b9d-afa0-49f8-b7af-e4e6a42ad299, Label: crop production principles, Job Count: 35, Job IDs: [141059, 151290, 156338, 156744, 184552, 1647466, 1657697, 1662745, 1663151, 1689159, 1702404, 1703236, 1706977, 1708081, 1756129, 1788922, 1793813, 1856862, 159328, 1665735, 1702319, 1709656, 1713772, 1819381, 1853395, 1854356, 1854878, 123574, 156301, 1630918, 1662708, 1709715, 1751210, 1807209, 1736333]
Skill ID: http://data.europa.eu/esco/skill/c4b03a17-a93e-4a8e-9846-4c4d469d4001, La

# Converting and saving relevant skills as csv file

In [ ]:
def save_skills_to_csv(sorted_skills):
    skill_data = []
    for skill_id, skill_info in sorted_skills:
        skill_data.append({
            'skill_label': skill_info['label'], 'skill_id': skill_id, 'job_count': skill_info['job_count'], 'job_ids': skill_info['job_ids']
        })

    df_relevant_skills = pd.DataFrame(skill_data)

    try:
      df_relevant_skills.to_csv('relevant_skills.csv', index=False)
      print("DataFrame 'df_relevant_skills' successfully saved to 'relevant_skills.csv'")

    except Exception as e:
      print(f"Error saving DataFrame to CSV: {e}")

save_skills_to_csv(sorted_skills)

DataFrame 'df_relevant_skills' successfully saved to 'relevant_skills.csv'


In [ ]:
import pandas as pd


df = pd.read_csv("/content/relevant_skills.csv")


df.head()


,skill_label,skill_id,job_count,job_ids
0,harvest crop,http://data.europa.eu/esco/skill/2e4e8114-e579...,35,"[32661, 52208, 63540, 80351, 91684, 109562, 11..."
1,crop production principles,http://data.europa.eu/esco/skill/4ca83b9d-afa0...,35,"[141059, 151290, 156338, 156744, 184552, 16474..."
2,agroecology,http://data.europa.eu/esco/skill/c4b03a17-a93e...,29,"[32661, 45646, 45647, 45653, 52208, 61744, 635..."
3,manage crop production,http://data.europa.eu/esco/skill/754de174-292f...,25,"[82355, 92925, 123970, 145804, 156728, 156744,..."
4,research improvement of crop yields,http://data.europa.eu/esco/skill/b47da85a-bfde...,25,"[156728, 156744, 184552, 1663135, 1663151, 168..."


In [ ]:
import pandas as pd

def save_jobs_to_csv(sorted_skills):
    if jobs_data.get('items'):
        df_jobs_data = pd.DataFrame(jobs_data['items'])
        try:
            df_jobs_data.to_csv('all_jobs_data.csv', index=False)
            print("DataFrame 'df_jobs_data' successfully saved to 'all_jobs_data.csv'")
        except Exception as e:
            print(f"Error saving DataFrame 'df_jobs_data' to CSV: {e}")
    else:
        print("jobs_data not found or is empty.")

def save_accepted_jobs_to_csv(accepted):
    if accepted:
        df_accepted_jobs = pd.DataFrame(list(accepted.values()))
        try:
            df_accepted_jobs.to_csv('accepted_jobs.csv', index=False)
            print("DataFrame 'df_accepted_jobs' successfully saved to 'accepted_jobs.csv'")
        except Exception as e:
            print(f"Error saving DataFrame 'df_accepted_jobs' to CSV: {e}")
    else:
        print("accepted not found or is empty.")

save_jobs_to_csv(sorted_skills)
save_accepted_jobs_to_csv(accepted)

DataFrame 'df_jobs_data' successfully saved to 'all_jobs_data.csv'
DataFrame 'df_accepted_jobs' successfully saved to 'accepted_jobs.csv'


In [ ]:
import pandas as pd

df = pd.read_csv("/content/all_jobs_data.csv")
print("--- all jobs ---")
display(df.head())

df1 = pd.read_csv("/content/accepted_jobs.csv")
print("\n--- accepted jobs ---")
display(df1.head())

--- all jobs ---


,skills,occupations,id,organization,title,description,experience_level,type,location,location_code,nuts1,nuts2,nuts3,upload_date,source,source_id
0,['http://data.europa.eu/esco/skill/435d1751-86...,['http://data.europa.eu/esco/occupation/1b5ccf...,1659,NaN,MANAGER OF PRODUCTION,"MILIORA FARM SA, which operates a modern sheep...",Entry / Junior,Full-time,"Larissos, Greece",EL,NaN,NaN,NaN,2024-09-03,kariera.gr,166734
1,['http://data.europa.eu/esco/skill/63f224ea-d8...,['http://data.europa.eu/esco/occupation/1b5ccf...,2638,NaN,Agricultural Sales - Export Managers,The Fertilizer Industry\nD. GAVRIEL & CO LTD\n...,Mid-level,Full-time,"Athens, Greece",EL,NaN,NaN,NaN,2024-08-29,kariera.gr,165914
2,['http://data.europa.eu/esco/skill/568e4759-c6...,['http://data.europa.eu/esco/occupation/08984b...,2893,NaN,Farm Veterinarian,"EPIROS SA, with a strong presence in the chees...",Senior,Full-time,"Larisa, Greece",EL,NaN,NaN,NaN,2024-08-28,kariera.gr,165616
3,['http://data.europa.eu/esco/skill/568e4759-c6...,['http://data.europa.eu/esco/occupation/08984b...,3237,NaN,Veterinary,"EPIROS SA, with a strong presence in the chees...",Senior,Full-time,"Ioannina, Greece",EL,NaN,NaN,NaN,2024-08-27,kariera.gr,165198
4,['http://data.europa.eu/esco/skill/0da516ee-e7...,['http://data.europa.eu/esco/occupation/11e6d3...,6744,NaN,Senior Consultant Agriculture and Food Sector F/M,"Position Description: You will join CGI, the w...",NaN,Full-time,"Rennes, France",FR,NaN,NaN,NaN,2024-09-05,lesjeudis,120054



--- accepted jobs ---


,skills,occupations,id,organization,title,description,experience_level,type,location,location_code,nuts1,nuts2,nuts3,upload_date,source,source_id
0,['http://data.europa.eu/esco/skill/435d1751-86...,['http://data.europa.eu/esco/occupation/1b5ccf...,1659,NaN,MANAGER OF PRODUCTION,"MILIORA FARM SA, which operates a modern sheep...",Entry / Junior,Full-time,"Larissos, Greece",EL,NaN,NaN,NaN,2024-09-03,kariera.gr,166734
1,['http://data.europa.eu/esco/skill/568e4759-c6...,['http://data.europa.eu/esco/occupation/08984b...,2893,NaN,Farm Veterinarian,"EPIROS SA, with a strong presence in the chees...",Senior,Full-time,"Larisa, Greece",EL,NaN,NaN,NaN,2024-08-28,kariera.gr,165616
2,['http://data.europa.eu/esco/skill/568e4759-c6...,['http://data.europa.eu/esco/occupation/08984b...,3237,NaN,Veterinary,"EPIROS SA, with a strong presence in the chees...",Senior,Full-time,"Ioannina, Greece",EL,NaN,NaN,NaN,2024-08-27,kariera.gr,165198
3,['http://data.europa.eu/esco/skill/0da516ee-e7...,['http://data.europa.eu/esco/occupation/11e6d3...,6744,NaN,Senior Consultant Agriculture and Food Sector F/M,"Position Description: You will join CGI, the w...",NaN,Full-time,"Rennes, France",FR,NaN,NaN,NaN,2024-09-05,lesjeudis,120054
4,['http://data.europa.eu/esco/skill/2e77d381-df...,['http://data.europa.eu/esco/occupation/1765f9...,28966,NaN,Versatile agricultural worker M/F,Agricultural Worker on permanent contract - Ag...,NaN,Full-time,"Castelnau-Magnoac, France",FR,NaN,NaN,NaN,2024-09-05,kariera.fr,1527768
